# Observability Overview

End-to-end demo of metrics, latency, and error counts across Agent.run, with_steps, and GraphRunner.

In [ ]:
import os
from getpass import getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

from agentic_codex import AgentBuilder, Context, GraphRunner, GraphNodeSpec, StepSpec, FunctionAdapter, EnvOpenAIAdapter
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.observability import Tracer

def stub_llm(prompt: str) -> str:
    return f"[stub llm] {prompt[:64]}"

try:
    llm = EnvOpenAIAdapter(model="gpt-4o-mini")
except Exception:
    llm = FunctionAdapter(stub_llm)

def simple_agent(name):
    def step(ctx: Context) -> AgentStep:
        content = f"{name}:{ctx.goal}:{ctx.scratch.get('batch_item','')}"
        return AgentStep(out_messages=[Message(role=name, content=content)], state_updates={name: True})
    return AgentBuilder(name=name, role=name).with_llm(llm).with_step(step).build()

# with_steps agent
with_steps_agent = (
    AgentBuilder(name="with-steps", role="orchestrator")
    .with_llm(llm)
    .with_steps(
        [
            StepSpec(name="prep", fn=simple_agent("prep").run),
            StepSpec(name="batch", fn=simple_agent("batch").run, batch_key="items", parallel=True, isolate_state=True),
        ]
    )
    .build()
)

graph = {
    "a": GraphNodeSpec(agent=with_steps_agent),
    "b": GraphNodeSpec(agent=simple_agent("b"), after=["a"]),
}

ctx = Context(goal="obs demo", scratch={"items": ["x", "y"]})
runner = GraphRunner(graph, allow_parallel=True, isolate_parallel_state=True)
result = runner.run(goal=ctx.goal, inputs=ctx.scratch)

metrics = ctx.scratch.get("_metrics", {})
len(result.messages), metrics